In [1]:
from langchain_core.tools import Tool
from langchain.tools import tool
from langchain_experimental.utilities import PythonREPL

C:\Users\Admin\AppData\Local\Temp\ipykernel_3028\3387063879.py:3: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL


In [2]:
python_repl = PythonREPL()

python_calculator = Tool(
    name="Python Calculator",
    func=python_repl.run,
    description="Useful when you want to perform operations usin python code, inputs must be Python codes ",
)

python_calculator.invoke("a=1;b=2;print(a+b)")

Python REPL can execute arbitrary code. Use with caution.


'3\n'

In [20]:
@tool
def weather_forecast(location: str) -> str:
    """ "shows weather condition"""
    return f"The weather for {location} today is currently sunny at 72`F"

In [ ]:
toolkit = [python_calculator, weather_forecast]

In [ ]:
# agent= model + harness

# 1st Agent
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()
API_KEY = os.getenv("OLLAMA_API_KEY")

if not API_KEY:
    raise ValueError("API KEY NOT FOUND")

llm = ChatOllama(
    model="gpt-oss:20b",
    base_url="https://ollama.com",
    temperature=0,
    client_kwargs={"headers": {"Authorization": f"Bearer {API_KEY}"}},
)


agent = create_agent(
    model=llm, tools=[weather_forecast], system_prompt="You are a helpful assistant"
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in San Fransisco?"}]}
)

print(result["messages"][-1].content_blocks[0]["text"])

The weather in San Francisco today is sunny with a temperature of about 72 °F.


In [61]:
# 2nd Agent

from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool

In [95]:
# Create the ReAct agent prompt template
# The ReAct prompt needs to instruct the model to follow the thought-action-observation pattern
# tools and tool-names,agent-scratchpad are predefined variables

# ReAct architecture


prompt_template = """You are an agent who has access to the following tools:

{tools}

The available tools are: {tool_names}

To use a tool, please use the following format:
```
Thought: I need to figure out what to do
Action: tool_name
Action Input: the input to the tool
```

After you use a tool, the observation will be provided to you:
```
Observation: result of the tool
```

Then you should continue with the thought-action-observation cycle until you have enough information to respond to the user's request directly.
When you have the final answer, respond in this format:
```
Thought: I know the answer
Final Answer: the final answer to the original query
```

Remember, when using the Python Calculator tool, the input must be valid Python code.

Begin!

Question: {input}
Thought:{agent_scratchpad}
Action:
"""

prompt = PromptTemplate.from_template(prompt_template)

In [96]:
# Create the agent
agent = create_react_agent(llm=llm, tools=toolkit, prompt=prompt)

In [97]:
#  Create the agent executor
agent_executor = AgentExecutor(
    agent=agent, tools=toolkit, verbose=True, handle_parsing_errors=True
)

In [99]:
result = agent_executor.invoke({"input": "What is the weather in Yangon"})
print(result["output"])



> Entering new AgentExecutor chain...
Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Invalid Format: Missing 'Action:' after 'Thought:'Thought: I need to get the current weather forecast for Yangon using the weather_forecast tool.  
Action: weather_forecast  
Action Input: Yangon  The weather for Yangon today is currently sunny at 72`F

> Finished chain.
Agent stopped due to iteration limit o